# 経済データ分析Ⅱ　第4回
# 係数の期待値と分散

第3回では回帰分析の仮定1〜6を説明し、誤差項と残差の違いを確かめた。今回はその続きである。第3回のデータを作り直し、仮定の下で OLS の係数の期待値と分散を導く。必要な説明変数を省いたときに係数がどれだけずれるかも、標本を取り直す実験で確かめる。仮定の番号は第3回と同じである。

## Notebook の使い方

コードセルを上から順に選び、`Shift + Enter` で実行します。途中のセルから実行して「変数が見つからない」というエラー（`NameError`）が出たら、先頭のセルから実行し直してください。

JupyterLite では **Python (Pyodide)**、ローカルの Jupyter では Python カーネルを使います。データは Notebook の中で作るので、CSV ファイルの準備は要りません。

図の英語は、`Estimated income coefficient`＝所得の係数の推定値、`Number of samples`＝標本の数、`Multiple regression`＝重回帰、`Simple regression (household size omitted)`＝世帯人数を省いた単回帰、`True income coefficient`＝所得の真の係数です。

## 1. 準備

次のセルで、今回使うライブラリを読み込む。`if sys.platform == 'emscripten':` から始まる部分は、ブラウザ（JupyterLite）で実行しているときに必要なライブラリを用意する処理である。`np`、`pd`、`plt`、`sm` は第2回・第3回と同じ略称で、それぞれ NumPy、pandas、Matplotlib、statsmodels を指す。

In [ ]:
import sys
if sys.platform == 'emscripten':
    import pyodide_js
    await pyodide_js.loadPackage(['numpy', 'pandas', 'matplotlib', 'statsmodels'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

## 2. 第3回のデータ

新しい Notebook には第3回で作った変数が残っていないので、次のセルで第3回の第2節と同じ200世帯のデータを作り直す。シードも同じ202610なので、第3回と同じ値になる。各行の働きは第3回で説明したとおりである。

消費は次の式で作られている。

$$消費_i=5+0.3\times 所得_i+2\times 世帯人数_i+u_i$$

誤差 $u_i$ は、平均0・標準偏差4万円の正規分布から世帯ごとに独立に選んでいる。

In [ ]:
rng = np.random.default_rng(202610)
n = 200
sizes = np.array([1, 2, 3, 4, 5])
share = np.array([0.380, 0.281, 0.166, 0.119, 0.054])
household_size = rng.choice(sizes, size=n, p=share)
mean_income = np.array([37.0, 54.1, 63.0, 72.2, 69.3])
multiplier = rng.lognormal(mean=-0.06, sigma=0.35, size=n)
income = mean_income[household_size - 1] * multiplier
true_error = rng.normal(0, 4, size=n)
consumption = 5 + 0.3 * income + 2 * household_size + true_error

df = pd.DataFrame({'income': income, 'household_size': household_size,
                   'consumption': consumption})
df.head().round(1)

`income` は月額所得（万円）、`household_size` は世帯人数（人）、`consumption` は月額消費（万円）である。第3回と同じく、所得と世帯人数で消費を説明する重回帰と、世帯人数を省いて所得だけで説明する単回帰を推定しておく。結果の `.params` に係数が入っている。

In [ ]:
X_multiple = sm.add_constant(df[['income', 'household_size']])
multiple = sm.OLS(df['consumption'], X_multiple).fit()
X_simple = sm.add_constant(df[['income']])
simple = sm.OLS(df['consumption'], X_simple).fit()
print(f"重回帰の所得の係数：{multiple.params['income']:.3f}")
print(f"所得だけの回帰の傾き：{simple.params['income']:.3f}")

重回帰の所得の係数は0.300で、真の値0.3に近い。所得だけの回帰の傾きは0.348で、第3回の4.2節で見たように、世帯人数による消費の差の一部まで所得の係数に取り込んでいる。このずれは第4節で調べる。

## 3. 係数の期待値と分散

この節では、第3回の第3.2節で挙げた性質のうち、不偏性と係数の分散の式を導く。説明変数が所得だけの単回帰で考え、5世帯の小さな例で式の意味を確かめる。

所得と世帯人数の値は固定して考える。標本を取り直すと誤差 $u_i$ だけが変わり、推定値も変わる。第4節の実験と同じ設定である。

### 期待値と分散の性質

途中で、統計学で学ぶ期待値と分散の性質を3つ使う。$a$ は定数、$u$ は確率変数である。

- 性質1：$E[au]=aE[u]$、$\mathrm{Var}(au)=a^2\mathrm{Var}(u)$
- 性質2：確率変数の和の期待値は、それぞれの期待値の和である。
- 性質3：独立な確率変数の和の分散は、それぞれの分散の和である。

サイコロの目で確かめておく。サイコロの目 $u$ は1〜6が同じ確率で出るので、期待値は $E[u]=(1+2+3+4+5+6)/6=3.5$ である。分散は期待値からの差の二乗の平均 $\mathrm{Var}(u)=E[(u-3.5)^2]$ で、次のセルで計算する。

In [ ]:
faces = np.array([1, 2, 3, 4, 5, 6])
print(f'期待値：{faces.mean():.3f}')
print(f'分散：{((faces - faces.mean())**2).mean():.3f}')

分散は2.917である。次に、サイコロを10万回振る実験で3つの性質を確かめる。`rng_dice.integers(1, 7, size=100000)` は、1以上7未満の整数（1〜6の目）を10万個作る。`u1` と `u2` は、別々に振った2つのサイコロの目である。`.var()` は分散を求める。

In [ ]:
rng_dice = np.random.default_rng(202612)
u1 = rng_dice.integers(1, 7, size=100000)
u2 = rng_dice.integers(1, 7, size=100000)
print(f'1個の目：期待値 {u1.mean():.2f}、分散 {u1.var():.2f}')
print(f'1個の目の2倍：期待値 {(2 * u1).mean():.2f}、分散 {(2 * u1).var():.2f}')
print(f'2個の目の和：期待値 {(u1 + u2).mean():.2f}、分散 {(u1 + u2).var():.2f}')

1個の目の期待値3.51と分散2.90は、上で計算した3.5と2.917に近い。目を2倍にすると、期待値は7.01で約2倍、分散は11.59で約4倍（$2^2$ 倍）になる（性質1）。2個の目の和の期待値は7.02で、3.5の2つ分にあたる（性質2）。和の分散は5.85で、2.917の2つ分に近い（性質3）。

期待値が同じ7でも、1個の目を2倍にした場合の分散は、2個の和の分散の約2倍ある。2個の和では、一方の大きい目ともう一方の小さい目が打ち消し合うことがあるからである。

3.1節以降では、確率変数 $u$ を誤差 $u_i$、定数 $a$ を重み $w_i$ として、この3つの性質を使う。

### 3.1. 単回帰の傾きと誤差

母集団の式に所得だけが入っている場合 $y_i=\beta_0+\beta_1x_{1i}+u_i$ を考える（世帯人数は第4節で加える）。OLS の傾きは

$$\hat\beta_1=\frac{\sum_{i=1}^{n}(x_{1i}-\bar x_1)(y_i-\bar y)}{\sum_{i=1}^{n}(x_{1i}-\bar x_1)^2}$$

である。$x_{1i}-\bar x_1$ は世帯 $i$ の所得が平均からどれだけ離れているかを表し、所得の**偏差**という。分母は偏差を2乗して全世帯で足した値で、所得の偏差の二乗和と呼んで $SST_1$ と書く（SST は total sum of squares の略）。所得が平均の周りに広く散らばっているほど、$SST_1$ は大きい。

この式を5世帯の小さな例で計算してみる。所得は20、30、40、50、60万円、真の係数は $\beta_0=5$、$\beta_1=0.3$ とし、誤差は2、−1、3、−4、1万円とする。例で使う変数の名前には `_ex` を付ける。

`np.array([...])` は数を並べた配列を作る。配列どうしの計算は、同じ位置の値どうしで行われる。

In [ ]:
x_ex = np.array([20, 30, 40, 50, 60])
u_ex = np.array([2, -1, 3, -4, 1])
y_ex = 5 + 0.3 * x_ex + u_ex
print(y_ex)

1世帯目の消費は $5+0.3\times20+2=13$ 万円である。`13.` の点は、小数を含む数として計算されたことを表す。

次に、所得の偏差 `dev_ex`、その二乗和 `sst_ex`、傾きを式どおりに計算する。

In [ ]:
dev_ex = x_ex - x_ex.mean()
sst_ex = (dev_ex**2).sum()
slope_ex = (dev_ex * (y_ex - y_ex.mean())).sum() / sst_ex
print(f'偏差：{dev_ex}')
print(f'SST：{sst_ex:.0f}')
print(f'傾き：{slope_ex:.3f}')

所得の平均は40万円なので、偏差は −20、−10、0、10、20 になる。偏差の二乗和は

$$SST_1=(-20)^2+(-10)^2+0^2+10^2+20^2=400+100+0+100+400=1000$$

である。傾きは0.250で、真の値0.3より小さい。

statsmodels の OLS でも同じ値になることを確かめる。`sm.add_constant(x_ex)` で切片の列を加えて推定すると、`params` の2番目が傾きになる。Python では先頭を0番目と数えるので、2番目は `[1]` で取り出す。

In [ ]:
fit_ex = sm.OLS(y_ex, sm.add_constant(x_ex)).fit()
print(f'statsmodels の傾き：{fit_ex.params[1]:.3f}')

推定値が0.3からずれた理由を、式を書き直して調べる。偏差の合計 $\sum(x_{1i}-\bar x_1)$ は0になる（例では $-20-10+0+10+20=0$）。そのため、分子は $\sum(x_{1i}-\bar x_1)y_i$ と書ける。ここに $y_i=\beta_0+\beta_1x_{1i}+u_i$ を代入すると

$$\sum(x_{1i}-\bar x_1)y_i=\beta_0\sum(x_{1i}-\bar x_1)+\beta_1\sum(x_{1i}-\bar x_1)x_{1i}+\sum(x_{1i}-\bar x_1)u_i$$

となる。右辺の3つの項と左辺を、例で計算する。

In [ ]:
print(f'第1項：{5 * dev_ex.sum():.0f}')
print(f'第2項：{0.3 * (dev_ex * x_ex).sum():.0f}')
print(f'第3項：{(dev_ex * u_ex).sum():.0f}')
print(f'左辺：{(dev_ex * y_ex).sum():.0f}')

第1項は、偏差の合計が0なので0になる。第2項は300で、$\beta_1=0.3$ に $\sum(x_{1i}-\bar x_1)x_{1i}=1000$ を掛けた値である。この1000は $SST_1$ に等しい。一般に、0である $\sum(x_{1i}-\bar x_1)\bar x_1$ を引くと $\sum(x_{1i}-\bar x_1)x_{1i}=\sum(x_{1i}-\bar x_1)^2=SST_1$ になるからである。第3項は −50 で、3つの項の合計250が左辺に一致する。

両辺を $SST_1$ で割ると

$$\hat\beta_1=\beta_1+\sum_{i=1}^{n}w_iu_i,\qquad w_i=\frac{x_{1i}-\bar x_1}{SST_1}$$

となる。例では $250/1000=0.3+(-50)/1000$ で、誤差の重み付きの和は $\sum w_iu_i=-0.05$ になる。推定値0.25は真の値0.3より0.05小さい。推定値は、母数 $\beta_1$ に誤差の重み付きの和を足したものである。

重み $w_i$ と $\sum w_iu_i$ を例で計算し、$0.3+\sum w_iu_i$ が推定値に一致することを確かめる。

In [ ]:
w_ex = dev_ex / sst_ex
print(f'重み：{w_ex}')
print(f'Σ w_i u_i：{(w_ex * u_ex).sum():.3f}')
print(f'0.3 + Σ w_i u_i：{0.3 + (w_ex * u_ex).sum():.3f}')

重み $w_i$ は所得だけで決まり、平均より所得の高い世帯では正、低い世帯では負になる。所得が平均の40万円の世帯は重みが0なので、その誤差3万円は推定値に影響しない。所得20万円の世帯（重み −0.02、誤差2万円）と所得50万円の世帯（重み0.01、誤差 −4万円）が、それぞれ推定値を0.04ずつ下げている。所得が平均から離れた世帯ほど、誤差が推定値を大きく動かす。

### 3.2. 期待値と不偏性

標本を取り直すと、誤差 $u_i$ が変わり、$\sum w_iu_i$ も変わる。重み $w_i$ は所得だけで決まるので変わらない。例で、誤差の符号をすべて逆にした標本（−2、1、−3、4、−1万円）を考える。

In [ ]:
u_ex2 = -u_ex
print(f'推定値：{0.3 + (w_ex * u_ex2).sum():.3f}')

推定値は0.350で、今度は真の値より0.05大きい。2つの標本の推定値の平均は0.3になる。

第3回の3.2節で説明したように、推定値が**不偏**であるとは、標本を取り直して推定を何度も繰り返したとき、推定値の平均が真の係数に一致することをいう。1回ごとの推定値は、例の0.25と0.35のように真の係数の上下にばらつく。

仮定4から各誤差の平均は0なので、性質1と性質2を使うと

$$E[\hat\beta_1]=\beta_1+\sum_{i=1}^{n}w_iE[u_i]=\beta_1$$

となる。推定値の平均が真の係数に一致するので、OLS の傾きは不偏である。仮定4が成り立たず、所得の高い世帯ほど誤差の平均が大きいと、正の重みの世帯で誤差が大きくなる。そのとき $\sum w_iE[u_i]$ は正になり、推定値は上にずれる。

### 3.3. 分散

推定値のばらつきを表す分散を求める。仮定2から誤差は互いに独立で、仮定5から分散はどれも $\sigma^2$ である。重み $w_i$ は定数なので、性質1と性質3から

$$\mathrm{Var}(\hat\beta_1)=\sum_{i=1}^{n}w_i^2\sigma^2=\frac{\sigma^2\sum(x_{1i}-\bar x_1)^2}{SST_1^2}=\frac{\sigma^2}{SST_1}$$

となる。例で、誤差の標準偏差を4万円（$\sigma^2=16$）として計算する。`np.sqrt()` は平方根を求める。

In [ ]:
print(f'Σ w_i²：{(w_ex**2).sum():.4f}')
print(f'1 / SST：{1 / sst_ex:.4f}')
print(f'推定値の標準偏差：{np.sqrt(16 / sst_ex):.3f}')

重みの二乗和は0.001で、$1/SST_1=1/1000$ に等しい。推定値の分散は $16/1000=0.016$ で、標準偏差は約0.126である。5世帯しかないので、推定値が真の値0.3から0.1以上ずれることも珍しくない。

同じ所得の世帯を2つずつにして、世帯の数を10に増やすと、$SST_1$ は2倍の2000になる。

In [ ]:
x_ten = np.array([20, 20, 30, 30, 40, 40, 50, 50, 60, 60])
sst_ten = ((x_ten - x_ten.mean())**2).sum()
print(f'SST：{sst_ten:.0f}')
print(f'推定値の標準偏差：{np.sqrt(16 / sst_ten):.3f}')

標準偏差は0.126から0.089に下がる。所得のばらつき $SST_1$ が大きいほど、また世帯数が多いほど、傾きのばらつきは小さい。

期待値の計算（3.2節）には仮定5を使っていないので、不偏性には仮定5が要らないことも分かる。

## 4. 標本を取り直したときの推定値の分布

所得と世帯人数を固定し、誤差 $u_i$ だけを1,000回作り直す。標本ごとに、重回帰の所得の係数と、世帯人数を省いて消費を所得だけに回帰した傾き $\tilde\beta_1$ を計算する。

重回帰では仮定1〜4が成り立つので、第3回の3.2節で説明したとおり推定値は不偏で、平均は母数0.3になる。所得だけの回帰の傾きは、3.1節と同じ計算で求められる。ただし今度は、母集団の式 $y_i=\beta_0+\beta_1x_{1i}+\beta_2x_{2i}+u_i$ に世帯人数が入っているので、代入すると世帯人数の項が残る。

$$\tilde\beta_1=\beta_1+\beta_2\,\frac{\sum(x_{1i}-\bar x_1)x_{2i}}{SST_1}+\sum_{i=1}^{n}w_iu_i$$

真ん中の分数は、世帯人数を所得に単回帰したときの傾きで、$\tilde\delta_1$ と書く。所得が1万円高い世帯で、世帯人数が平均的に何人多いかを表す。最後の項の期待値は仮定4から0なので

$$E[\tilde\beta_1]=\beta_1+\beta_2\tilde\delta_1$$

となる。$\tilde\delta_1$ を計算し、所得だけの回帰の傾きの平均を予想する。

In [ ]:
size_on_income = sm.OLS(df['household_size'], X_simple).fit().params['income']
print(f'世帯人数を所得に回帰した傾き：{size_on_income:.4f}')
print(f'所得だけの回帰の傾きの平均（予想）：{0.3 + 2 * size_on_income:.3f}')

傾きは約0.024で、所得が10万円高い世帯は、世帯人数が平均0.24人多い。所得が同じなら世帯人数が1人多いと消費は2万円多いので、所得が10万円高い世帯の消費は、世帯人数の違いを通じて $2\times0.24\approx0.49$ 万円多くなる。所得1万円当たりでは約0.05で、これが所得の係数に加わる。

所得だけの回帰の傾きの平均は $0.3+2\times0.024\approx0.35$ と予想できる。必要な説明変数を省いたことで生じるこの系統的なずれ（ここでは約0.05）を**欠落変数バイアス**という。

**実行前の予想**：重回帰と所得だけの回帰の所得の係数は、それぞれどの値の周りに分布するでしょうか。どちらの分布が広いでしょうか。

下のセルでは、空のリスト `slopes_multiple` と `slopes_simple` を用意し、誤差を作り直すたびに、重回帰の所得の係数と所得だけの回帰の傾きを `.append()` で1つずつ加える。`range(1000)` は1,000回繰り返す指定で、何回目かの番号は使わないので、変数名を `_` にしている。

In [ ]:
rng_repeat = np.random.default_rng(202611)
slopes_multiple = []
slopes_simple = []
for _ in range(1000):
    y_repeat = 5 + 0.3 * income + 2 * household_size + rng_repeat.normal(0, 4, n)
    slopes_multiple.append(sm.OLS(y_repeat, X_multiple).fit().params['income'])
    slopes_simple.append(sm.OLS(y_repeat, X_simple).fit().params['income'])

2つのリストを表にまとめ、`.mean()` で平均を、`.std()` で標準偏差を求める。

In [ ]:
results = pd.DataFrame({'multiple': slopes_multiple, 'simple': slopes_simple})
print(f"重回帰の所得の係数：平均 {results['multiple'].mean():.3f}、標準偏差 {results['multiple'].std():.4f}")
print(f"所得だけの回帰の傾き：平均 {results['simple'].mean():.3f}、標準偏差 {results['simple'].std():.4f}")

2つの推定値の分布をヒストグラムに描く。

- `plt.subplots()` で図を用意する。`fig` は図全体、`ax` は図を描く場所である。
- `np.linspace(0.24, 0.40, 41)` は、0.24から0.40までを40の区間に等しく分ける境目を作る。2つのヒストグラムで同じ区間を使うように、`bins` に渡す。
- `ax.hist()` はヒストグラムを描く。`alpha=0.6` は色の濃さで、重なった部分も見えるように薄くしている。`label` は凡例に出す名前である。
- `ax.axvline(0.3, color='red', linestyle='--')` は、横軸0.3の位置に赤い破線を引く。
- `ax.set()` で横軸（`xlabel`）と縦軸（`ylabel`）の名前を付ける。`ylim=(0, 200)` は縦軸の範囲で、凡例が棒に重ならないように上を空けている。
- `ax.legend(loc='upper left')` で、凡例を左上に表示する。

最後の `plt.show()` で図を表示する。

In [ ]:
fig, ax = plt.subplots()
bins = np.linspace(0.24, 0.40, 41)
ax.hist(slopes_multiple, bins=bins, alpha=0.6, label='Multiple regression')
ax.hist(slopes_simple, bins=bins, alpha=0.6, label='Simple regression (household size omitted)')
ax.axvline(0.3, color='red', linestyle='--', label='True income coefficient')
ax.set(xlabel='Estimated income coefficient', ylabel='Number of samples', ylim=(0, 200))
ax.legend(loc='upper left')
plt.show()

重回帰の推定値は0.3の周りに、所得だけの回帰の推定値は約0.35の周りに分布する。所得だけの回帰の分布は少し狭いが、中心は真の係数から離れている。分散の小ささだけでは、推定方法の良し悪しを判断できない。

1,000回の平均や標準偏差も乱数によって変動する。この実験は、仮定の下で成り立つ性質を数値で確かめたものである。

## 関数の資料

- [statsmodels：OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html)
- [NumPy：乱数の生成（Generator）](https://numpy.org/doc/stable/reference/random/generator.html)